In [2]:
!pip install -q transformers torch scipy

In [7]:
# Install Piper TTS (pre-compiled, no C++ build required)
!pip install -q piper-tts

# Download the Nepali VITS/FastSpeech ONNX model and configuration
!wget -q https://huggingface.co/ampixa/real-nepali-v0.2-kala/resolve/main/real_nepali_v02_kala.fp32.onnx -O /content/model.onnx
!wget -q https://huggingface.co/ampixa/real-nepali-v0.2-kala/resolve/main/real_nepali_v02_kala.fp32.onnx.json -O /content/model.onnx.json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.7 MB/s eta 0:00:00


In [10]:
# 1. Download the correct Piper binary for Google Colab (Linux x86_64)
!wget -q https://github.com/rhasspy/piper/releases/download/2023.11.14-2/piper_linux_x86_64.tar.gz
!tar -xzf piper_linux_x86_64.tar.gz

# 2. Download the Nepali model components explicitly into the main directory
!wget -q https://huggingface.co/ampixa/real-nepali-v0.2-kala/resolve/main/real_nepali_v02_kala.fp32.onnx -O /content/model.onnx
!wget -q https://huggingface.co/ampixa/real-nepali-v0.2-kala/resolve/main/real_nepali_v02_kala.fp32.onnx.json -O /content/model.onnx.json

In [3]:
from huggingface_hub import login

login(token="YOUR_HF_TOKEN_HERE")

In [13]:
# 1. Clone the custom Nepali text frontend repository
!git clone https://github.com/Ampixa/nepa-newa-text-frontend

# 2. Install the necessary Python backend packages
!pip install -q onnxruntime huggingface_hub numpy

Cloning into 'nepa-newa-text-frontend'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 94 (delta 13), reused 90 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 7.74 MiB | 17.74 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [16]:
import os
import sys
import json
import time
import subprocess

# 1. Install the official kala-tts package from PyPI
print("Installing kala-tts globally...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kala-tts"])
print("Environment setup complete!\n")

# -------------------------------------------------------------------
# THE EVALUATION LOOP
# -------------------------------------------------------------------
JSON_PATH = "/content/nepali_evaluation_set.json"
OUTPUT_DIR = "/content/output_audio"

def test_fast_model():
    if not os.path.exists(JSON_PATH):
        print(f"ERROR: Cannot find '{JSON_PATH}'. Make sure it is in your side panel.")
        return

    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Loaded {len(dataset)} evaluation samples. Starting audio generation...\n")
    print("-" * 50)

    generated_count = 0

    for item in dataset:
        item_id = item["ID"]
        eval_set = item["EVALUATION SET"]
        text = item["TEXT"]

        filename = f"{OUTPUT_DIR}/{item_id}_{eval_set}.wav"
        print(f"Processing ID   : {item_id}")

        start_time = time.time()
        try:
            # Because it is installed via pip, the module is globally available!
            process = subprocess.Popen(
                [sys.executable, "-m", "kala_tts", text, "-o", filename],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )
            stdout, stderr = process.communicate()

            if process.returncode != 0:
                print(f"Status          : FAILED")
                print(f"Error Output    : {stderr.decode('utf-8').strip()}")
            else:
                elapsed = time.time() - start_time
                generated_count += 1
                print(f"Status          : Success (Saved to {filename})")
                print(f"Generation Time : {elapsed:.4f} seconds")

        except Exception as e:
            print(f"Status          : FAILED")
            print(f"Exception       : {str(e)}")

        print("-" * 50)

    print(f"\nTesting Complete!")
    print(f"Successfully generated {generated_count}/{len(dataset)} files.")

if __name__ == "__main__":
    test_fast_model()

Installing kala-tts globally...
Environment setup complete!

Loaded 20 evaluation samples. Starting audio generation...

--------------------------------------------------
Processing ID   : NEP_01
Status          : Success (Saved to /content/output_audio/NEP_01_velars_gutturals.wav)
Generation Time : 3.6378 seconds
--------------------------------------------------
Processing ID   : NEP_02
Status          : Success (Saved to /content/output_audio/NEP_02_palatals_and_trills.wav)
Generation Time : 4.3836 seconds
--------------------------------------------------
Processing ID   : NEP_03
Status          : Success (Saved to /content/output_audio/NEP_03_retroflexes_and_nasalization.wav)
Generation Time : 3.3796 seconds
--------------------------------------------------
Processing ID   : NEP_04
Status          : Success (Saved to /content/output_audio/NEP_04_labials_and_nasalization.wav)
Generation Time : 3.4073 seconds
--------------------------------------------------
Processing ID   : NEP

In [17]:
!pip install -q openai-whisper jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.8 MB/s eta 0:00:00


In [18]:
import os
import json
import whisper
import jiwer
import shutil
from google.colab import files

# 1. Configuration and Paths
JSON_PATH = "/content/nepali_evaluation_set.json"
OUTPUT_DIR = "/content/output_audio"
ZIP_PATH = "/content/nepali_audio_evaluation" # Without the .zip extension

def evaluate_and_download():
    # 2. Load the Whisper Medium model
    print("Loading Whisper 'medium' model... (This may take a minute)")
    model = whisper.load_model("medium")

    # 3. Load the dataset for Ground Truth text
    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    ground_truths = []
    hypotheses = []

    print("\nStarting Speech-to-Text Transcription...\n")
    print("-" * 50)

    for item in dataset:
        item_id = item["ID"]
        eval_set = item["EVALUATION SET"]
        reference_text = item["TEXT"]

        filename = f"{OUTPUT_DIR}/{item_id}_{eval_set}.wav"

        if not os.path.exists(filename):
            print(f"WARNING: File missing for {item_id}, skipping...")
            continue

        print(f"Transcribing : {item_id} ({eval_set})")

        # Transcribe the generated audio, forcing the language to Nepali
        result = model.transcribe(filename, language="ne")
        transcribed_text = result["text"].strip()

        # Store for evaluation
        ground_truths.append(reference_text)
        hypotheses.append(transcribed_text)

        print(f"Reference    : {reference_text}")
        print(f"Whisper Heard: {transcribed_text}")
        print("-" * 50)

    # 4. Calculate Error Rates using jiwer
    if ground_truths and hypotheses:
        wer = jiwer.wer(ground_truths, hypotheses)
        cer = jiwer.cer(ground_truths, hypotheses)

        print(f"\n✅ Evaluation Complete!")
        print(f"Word Error Rate (WER)      : {wer * 100:.2f}%")
        print(f"Character Error Rate (CER) : {cer * 100:.2f}%")
    else:
        print("\n❌ No audio files were found to evaluate.")

    # 5. Zip the directory and trigger download
    print(f"\nZipping the '{OUTPUT_DIR}' folder...")
    shutil.make_archive(ZIP_PATH, 'zip', OUTPUT_DIR)

    print("Prompting browser download...")
    files.download(f"{ZIP_PATH}.zip")

if __name__ == "__main__":
    evaluate_and_download()

Loading Whisper 'medium' model... (This may take a minute)


100%|██████████████████████████████████████| 1.42G/1.42G [00:14<00:00, 104MiB/s]



Starting Speech-to-Text Transcription...

--------------------------------------------------
Transcribing : NEP_01 (velars_gutturals)
Reference    : कागती खाएर केटाकेटीहरू खुसी हुँदै घर गए।
Whisper Heard: अकादाती फायर केता केती हरू कुस्सी हूदे गार गये।
--------------------------------------------------
Transcribing : NEP_02 (palatals_and_trills)
Reference    : चराहरू चिरबिर गर्दै चाँडै उडेर चौतारीमा बसे।
Whisper Heard: सराहरो चिवीर करते चान्डे उरे और छोतारी मा बसे।
--------------------------------------------------
Transcribing : NEP_03 (retroflexes_and_nasalization)
Reference    : ठूलो डाँडामाथि ढकमक्क गुराँस फुल्दा धेरै राम्रो देखिन्छ।
Whisper Heard: तुप्लोर डडा माति तकमक कगुरास फूलडा देरे राम्मु देखिंजा।
--------------------------------------------------
Transcribing : NEP_04 (labials_and_nasalization)
Reference    : फराकिलो बाटोमा भाइ र बहिनी पानी पिउँदै हिँडे।
Whisper Heard: फरा किलो बातव मा भाहर भहीने पाने पीव गे हिन्धे।
--------------------------------------------------
Transcr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>